In [ ]:
from src.ingestion.loader import AmazonReviewLoader

loader = AmazonReviewLoader(
    "datasets/raw/reviews_Tools_and_Home_Improvement.json.gz"
)

for i, review in enumerate(loader.stream()):

    print(review)

    if i == 2:
        break

In [ ]:
from src.ingestion.loader import AmazonReviewLoader
from src.ingestion.mapper import AmazonReviewMapper


loader = AmazonReviewLoader(
    "datasets/raw/reviews_Tools_and_Home_Improvement.json.gz"
)

record = next(loader.stream())

mapped = AmazonReviewMapper.map(record)

for k, v in mapped.items():
    print(f"{k:20} : {v}")

In [ ]:
from src.ingestion.loader import AmazonReviewLoader
from src.ingestion.mapper import AmazonReviewMapper
from src.ingestion.validator import RetailReviewValidator

loader = AmazonReviewLoader(
    "datasets/raw/reviews_Tools_and_Home_Improvement.json.gz"
)

for review in loader.stream():

    mapped = AmazonReviewMapper.map(review)

    valid, reason = RetailReviewValidator.validate(mapped)

    print(valid, reason)

    break

In [ ]:
from pprint import pprint

from src.ingestion.loader import AmazonReviewLoader
from src.ingestion.mapper import AmazonReviewMapper
from src.ingestion.validator import RetailReviewValidator
from src.ingestion.profiler import DataProfiler

loader = AmazonReviewLoader(
    "datasets/raw/reviews_Tools_and_Home_Improvement.json.gz"
)

profiler = DataProfiler()

for review in loader.stream():

    mapped = AmazonReviewMapper.map(review)

    valid, _ = RetailReviewValidator.validate(mapped)

    if valid:
        profiler.update(mapped)

pprint(profiler.report())

In [ ]:
from src.preprocessing.text_cleaner import TextCleaner
review = """
This was &#34;Amazing&#34;.

Visit https://amazon.com

Email me at abc@gmail.com
"""

print(TextCleaner.clean(review))

In [ ]:
from src.preprocessing.feature_engineering import FeatureEngineer
review = {
    "review_text": "This drill is amazing! Would definitely recommend it.",
    "helpful_yes": 8,
    "helpful_total": 10,
}

review = FeatureEngineer.generate(review)

for k, v in review.items():
    print(f"{k:<25} {v}")

In [ ]:
import gzip

with gzip.open(
    "datasets/raw/meta_Tools_and_Home_Improvement.json.gz",
    "rt",
    encoding="utf-8"
) as f:

    for i in range(3):
        print(f.readline())

In [ ]:
from src.ingestion.metadata_joiner import MetadataJoiner

review = {
    "product_id": "001212835X",
    "review_text": "Excellent lamp."
}

metadata = [
    {
        "product_id": "001212835X",
        "product_name": "Everett's Cottage Table Lamp",
        "category": "Lighting",
        "brand": "Everett",
        "image_url": "...",
        "description": "...",
        "features": []
    }
]

joiner = MetadataJoiner(metadata)

review = joiner.enrich(review)

print(review)

In [ ]:
from src.pipeline.retail_data_pipeline import RetailDataPipeline

pipeline = RetailDataPipeline(
    review_path="datasets/raw/reviews_Tools_and_Home_Improvement.json.gz",
    metadata_path="datasets/raw/meta_Tools_and_Home_Improvement.json.gz",
    output_path="datasets/processed/master_reviews.parquet",
)

df = pipeline.build()

print(df.head())

In [ ]:
from src.ingestion.meta_data_loader import MetadataLoader

loader = MetadataLoader(
    "datasets/raw/meta_Tools_and_Home_Improvement.json.gz"
)

for i, record in enumerate(loader.stream()):
    print(record.keys())

    print(record)

    if i == 2:
        break

In [ ]:
from src.ingestion.meta_data_loader import MetadataLoader

loader = MetadataLoader(
    "datasets/raw/meta_Tools_and_Home_Improvement.json.gz"
)

count = 0
total = 0

for record in loader.stream():

    total += 1

    if "feature" in record:
        count += 1

print("Products:", total)
print("Products with feature:", count)

In [ ]:
from src.ingestion.meta_data_loader import MetadataLoader

loader = MetadataLoader(
    "datasets/raw/meta_Tools_and_Home_Improvement.json.gz"
)

record = next(loader.stream())

print(record.keys())

In [ ]:
from pprint import pprint

from src.training.sentiment.tokenizer_analyzer import (
    TokenizerAnalyzer,
)

analyzer = TokenizerAnalyzer(
    "datasets/processed/master_reviews.parquet"
)

report = analyzer.analyze()

print("\n===== TOKEN REPORT =====")

pprint(report)

In [ ]:
from src.training.sentiment.data_splitter import DataSplitter

splitter = DataSplitter(
    "datasets/processed/master_reviews.parquet"
)

train_df, val_df, test_df = splitter.split(
    output_dir="datasets/splits"
)


In [ ]:
from transformers import AutoTokenizer

from src.training.sentiment.dataset import RetailSentimentDataset

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

dataset = RetailSentimentDataset(
    parquet_path="datasets/splits/train.parquet",
    tokenizer=tokenizer,
    max_length=256,
)

sample = dataset[0]

print(sample["input_ids"].shape)
print(sample["attention_mask"].shape)
print(sample["label"])

In [ ]:
from transformers import AutoTokenizer

from src.training.sentiment.dataset import RetailSentimentDataset
from src.training.sentiment.dataloader import RetailDataLoaderFactory

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

dataset = RetailSentimentDataset(
    parquet_path="datasets/splits/train.parquet",
    tokenizer=tokenizer,
    max_length=256,
)

loader = RetailDataLoaderFactory.create(
    dataset,
    batch_size=8,
    shuffle=True,
)

batch = next(iter(loader))

print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["label"].shape)

In [ ]:
from src.training.sentiment.metrics import SentimentMetrics
import numpy as np

labels = np.array([0, 1, 2, 2, 1, 0])

predictions = np.array([0, 1, 2, 1, 1, 0])

metrics = SentimentMetrics.compute(
    labels,
    predictions,
)

print(metrics)

print(
    SentimentMetrics.classification_report(
        labels,
        predictions,
    )
)

print(
    SentimentMetrics.confusion_matrix(
        labels,
        predictions,
    )
)

In [ ]:
from pprint import pprint

from src.training.sentiment.class_weights import (
    ClassWeightCalculator,
)

weights = ClassWeightCalculator.compute(
    "datasets/splits/train.parquet"
)

pprint(weights)

In [ ]:
from src.training.device import get_device
from src.training.sentiment.class_weights import ClassWeightCalculator
from src.training.sentiment.losses import LossFactory

device = get_device()

weights = ClassWeightCalculator.compute_tensor(
    "datasets/splits/train.parquet",
    device=device,
)

print(weights)

criterion = LossFactory.cross_entropy(
    class_weights=weights
)

print(criterion)

In [ ]:
from transformers import AutoTokenizer

from src.training.sentiment.dataset import RetailSentimentDataset
from src.training.sentiment.dataloader import RetailDataLoaderFactory

from src.training.sentiment.trainer import RetailSentimentTrainer
from src.training.sentiment.optimizer import OptimizerFactory
from src.training.sentiment.class_weights import ClassWeightCalculator
from src.training.sentiment.losses import LossFactory

from src.training.sentiment.config import SentimentConfig

from src.training.device import get_device


device = get_device()

tokenizer = AutoTokenizer.from_pretrained(
    SentimentConfig.MODEL_NAME
)

dataset = RetailSentimentDataset(
    parquet_path=SentimentConfig.TRAIN_DATA,
    tokenizer=tokenizer,
    max_length=SentimentConfig.MAX_LENGTH,
)

loader = RetailDataLoaderFactory.create(
    dataset,
    batch_size=8,
    shuffle=True,
)

trainer = RetailSentimentTrainer(
    model_name=SentimentConfig.MODEL_NAME,
    num_labels=SentimentConfig.NUM_CLASSES,
)

weights = ClassWeightCalculator.compute_tensor(
    SentimentConfig.TRAIN_DATA,
    device,
)

criterion = LossFactory.cross_entropy(
    weights,
)

optimizer = OptimizerFactory.adamw(
    trainer.model,
    learning_rate=SentimentConfig.LEARNING_RATE,
    weight_decay=SentimentConfig.WEIGHT_DECAY,
)

batch = next(iter(loader))

loss = trainer.train_one_batch(
    batch,
    optimizer,
    criterion,
)

print(loss)

In [ ]:
from src.training.sentiment.model_factory import ModelFactory

from src.training.sentiment.config import (
    SentimentConfig,
)

model = ModelFactory.create(

    model_name=SentimentConfig.MODEL_NAME,

    num_labels=SentimentConfig.NUM_CLASSES,

    id2label=SentimentConfig.ID2LABEL,

    label2id=SentimentConfig.LABEL2ID,
)

print(type(model))

In [1]:
train

NameError: name 'train' is not defined

In [2]:
import pandas as pd

train = pd.read_parquet("datasets/splits/train.parquet")
val = pd.read_parquet("datasets/splits/validation.parquet")

train.head(500).to_parquet(
    "datasets/splits/train_small.parquet",
    index=False,
)

val.head(200).to_parquet(
    "datasets/splits/validation_small.parquet",
    index=False,
)

print("Done")

Done


In [7]:
train.columns

Index(['review_id', 'source', 'product_id', 'customer_id', 'customer_name',
       'review_title', 'review_text', 'rating', 'helpful_yes', 'helpful_total',
       'review_timestamp', 'review_date', 'sentiment_label', 'language',
       'product_name', 'department', 'category', 'subcategory',
       'category_path', 'image_url', 'character_count', 'word_count',
       'sentence_count', 'average_word_length', 'contains_question',
       'contains_exclamation', 'helpful_ratio'],
      dtype='str')

In [5]:
train[0:1].head()

array([['4557011f-1e1e-49e9-9310-50946063bd31', 'amazon', 'B002FISGY2',
        'A2R3ZPJE8TZJXK', 'R. Barbre',
        "Don't buy - - It isn't accurate.",
        'i loved the idea.. but it ended up being a total waste of my money.mine is 1/16" off... and there is no way to adjust it. plus you have to "recalibrate" it if it sits for more then a few days - which i actually would not mind, if it calibrated to be accurate!so, every time i use the digital part, i have to add 1/16 to what ever the screen says... considering i bought it for the "digital accuracy" that is totally annoying. since it is useless as a digital measure - $35 is a ridiculous price for a standard tape measure that looks cool.i will not be buying another, nor will i recommend to anyone. in fact, i will be telling others not to buy one.',
        2.0, 6, 6, 1266537600, '02 19, 2010', 'negative', None,
        'Digital Tape Measure', 'Power & Hand Tools',
        'Measuring & Layout Tools', 'Tape Measures',
        'Too

In [ ]:
import pandas as pd

train = pd.read_parquet("datasets/splits/train.parquet")
val = pd.read_parquet("datasets/splits/validation.parquet")

train.head(500).to_parquet(
    "datasets/splits/train_small.parquet",
    index=False,
)

val.head(200).to_parquet(
    "datasets/splits/validation_small.parquet",
    index=False,
)

print("Done")

from transformers import AutoTokenizer

from src.training.sentiment.dataset import RetailSentimentDataset
from src.training.sentiment.dataloader import RetailDataLoaderFactory
from src.training.sentiment.config import SentimentConfig

tokenizer = AutoTokenizer.from_pretrained(
    SentimentConfig.MODEL_NAME
)

train_dataset = RetailSentimentDataset(
    parquet_path="datasets/splits/train_small.parquet",
    tokenizer=tokenizer,
    max_length=SentimentConfig.MAX_LENGTH,
)

validation_dataset = RetailSentimentDataset(
    parquet_path="datasets/splits/validation_small.parquet",
    tokenizer=tokenizer,
    max_length=SentimentConfig.MAX_LENGTH,
)

train_loader = RetailDataLoaderFactory.create(
    train_dataset,
    batch_size=8,
    shuffle=True,
)

validation_loader = RetailDataLoaderFactory.create(
    validation_dataset,
    batch_size=8,
    shuffle=False,
)

from src.training.device import get_device

from src.training.sentiment.model_factory import ModelFactory
from src.training.sentiment.optimizer import OptimizerFactory
from src.training.sentiment.scheduler import SchedulerFactory
from src.training.sentiment.class_weights import (
    ClassWeightCalculator,
)
from src.training.sentiment.losses import LossFactory
from src.training.sentiment.trainer import (
    TransformerTrainer,
)

device = get_device()

model = ModelFactory.create(
    model_name=SentimentConfig.MODEL_NAME,
    num_labels=SentimentConfig.NUM_CLASSES,
    id2label=SentimentConfig.ID2LABEL,
    label2id=SentimentConfig.LABEL2ID,
)

model.to(device)

weights = ClassWeightCalculator.compute_tensor(
    "datasets/splits/train_small.parquet",
    device=device,
)

criterion = LossFactory.cross_entropy(weights)

optimizer = OptimizerFactory.adamw(
    model=model,
    learning_rate=SentimentConfig.LEARNING_RATE,
    weight_decay=SentimentConfig.WEIGHT_DECAY,
)

total_steps = (
    len(train_loader)
    * 1
)

scheduler = SchedulerFactory.linear_warmup(
    optimizer=optimizer,
    total_training_steps=total_steps,
    warmup_ratio=SentimentConfig.WARMUP_RATIO,
)

trainer = TransformerTrainer(
    model=model,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    tokenizer=tokenizer,
    device=device,
)

trainer.fit(
    train_loader=train_loader,
    validation_loader=validation_loader,
    epochs=1,
)

In [ ]:
from pathlib import Path
from copy import deepcopy

from src.training.sentiment.config import SentimentConfig


class SmallConfig(SentimentConfig):

    TRAIN_DATA = Path("datasets/splits/train_small.parquet")

    VALIDATION_DATA = Path("datasets/splits/validation_small.parquet")

    TEST_DATA = Path("datasets/splits/test_small.parquet")

    BATCH_SIZE = 8

    NUM_EPOCHS = 1

from src.training.sentiment.train import main

main(
    resume=False,
    config=SmallConfig,
)

In [1]:
from src.inference.sentiment.inference_pipeline import (
    SentimentInferencePipeline,
)

pipeline = SentimentInferencePipeline(
    model_directory="artifacts/sentiment/best_model",
)

result = pipeline.predict(
    "This drill is excellent. Battery life is amazing."
)

print(result)

/Users/prakash/envs/retail/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 12077.98it/s]


Prediction(label='positive', confidence=0.3925)


In [8]:
from src.nlp.retail_ner.build_weak_labels import WeakLabelBuilder

builder = WeakLabelBuilder(
    input_file="datasets/splits/train_small.parquet",
    output_file="datasets/ner/ner_train_small.parquet",
)

builder.build()

/Users/prakash/envs/retail/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 500/500 [00:00<00:00, 1100.57it/s]


In [9]:
import pandas as pd

df = pd.read_parquet(
    "datasets/ner/ner_train_small.parquet"
)

df.head()

,review_id,tokens,ner_tags
0,4557011f-1e1e-49e9-9310-50946063bd31,"[i, loved, the, idea, ., ., but, it, ended, up...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
1,8bae6a3b-6c73-4776-ae38-3343af7c3f89,"[the, 3, /, 16ths, seems, to, bore, a, slightl...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
2,e3092acd-cac4-4202-bce1-e801127f89fd,"[this, light, was, a, perfect, match, to, a, c...","[O, O, O, O, O, O, O, O, B-PRODUCT, O, O, O, O..."
3,944a131b-f281-4704-a7aa-67cf532eb3fb,"[it, ', s, a, good, looking, and, sturdy, post...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
4,f093192d-5dba-4bf1-a3ad-f0bffbe02645,"[can, ', t, say, enough, about, this, tool, .,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."


In [10]:
from src.nlp.retail_ner.predictor import RetailNERPredictor

predictor = RetailNERPredictor(
    "artifacts/retail_ner/best_model"
)

predictor.predict(
    "The Milwaukee drill battery stopped charging after one week."
)

OSError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': 'artifacts/retail_ner/best_model'. Use `repo_type` argument if needed.

In [ ]:
SmallRetailNERConfig

In [1]:
from src.nlp.retail_ner.train import main

from pathlib import Path

from src.nlp.retail_ner.config import RetailNERConfig


class SmallRetailNERConfig(RetailNERConfig):

    TRAIN_DATA = Path(
        "datasets/ner/ner_train_small.parquet"
    )

    VALIDATION_DATA = Path(
        "datasets/ner/ner_validation_small.parquet"
    )

    TEST_DATA = Path(
        "datasets/ner/ner_test_small.parquet"
    )

    BATCH_SIZE = 8

    NUM_EPOCHS = 1
    

main(
    resume=False,
    config=SmallRetailNERConfig,
)

/Users/prakash/envs/retail/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-01 00:34:24,346 | INFO | Using device: mps
2026-07-01 00:34:24,347 | INFO | Loading tokenizer: bert-base-uncased
2026-07-01 00:34:24,605 | INFO | HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
2026-07-01 00:34:24,607 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-07-01 00:34:24,849 | INFO | HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-07-01 00:34:25,148 | INFO | HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=


Epoch 1/1


2026-07-01 00:35:29,692 | INFO | Training Loss : 0.4128                                    
Epoch 1 Validation:   0%|          | 0/25 [00:00<?, ?it/s]Python(3957) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(3958) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(3963) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(3964) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
/Users/prakash/envs/retail/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/prakash/envs/retail/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in la


Train Loss      : 0.4128
Validation Loss : 0.0593
Accuracy         : 0.9915
Precision        : 0.0000
Recall           : 0.0000
Entity F1        : 0.0000
Learning Rate    : 0.00e+00


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.22it/s]
2026-07-01 00:35:46,255 | INFO | Checkpoint saved at artifacts/retail_ner/best_model
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.35it/s]
2026-07-01 00:35:48,202 | INFO | Checkpoint saved at artifacts/retail_ner/last_model
2026-07-01 00:35:48,210 | INFO | Training history saved.
2026-07-01 00:35:48,210 | INFO | Training Completed.



FINAL TEST EVALUATION


Epoch 1 Validation:   0%|          | 0/25 [00:00<?, ?it/s]Python(3987) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(3992) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(3997) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(3998) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
/Users/prakash/envs/retail/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/prakash/envs/retail/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _w


Evaluation
Loss      : 0.0761
Accuracy  : 0.9882
Precision : 0.0000
Recall    : 0.0000
Entity F1 : 0.0000


In [14]:
! pip install seqeval

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16252 sha256=34a97c521a0ad6ea2f725b427426930ba17508c0ed16aa14450c975ded57036a
  Stored in directory: /Users/prakash/Library/Caches/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [1]:
from src.nlp.retail_ner.build_weak_labels import WeakLabelBuilder

WeakLabelBuilder(
    input_file="datasets/splits/train_small.parquet",
    output_file="datasets/ner/ner_train_small.parquet",
).build()

WeakLabelBuilder(
    input_file="datasets/splits/validation_small.parquet",
    output_file="datasets/ner/ner_validation_small.parquet",
).build()

WeakLabelBuilder(
    input_file="datasets/splits/test_small.parquet",
    output_file="datasets/ner/ner_test_small.parquet",
).build()

/Users/prakash/envs/retail/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 200/200 [00:00<00:00, 1221.29it/s]


In [3]:
import pandas as pd

df = pd.read_parquet("datasets/ner/ner_train_small.parquet")

print(type(df.iloc[0]["tokens"]))
print(df.iloc[0]["tokens"])

print(type(df.iloc[0]["ner_tags"]))
print(df.iloc[0]["ner_tags"])

<class 'numpy.ndarray'>
['i' 'loved' 'the' 'idea' '.' '.' 'but' 'it' 'ended' 'up' 'being' 'a'
 'total' 'waste' 'of' 'my' 'money' '.' 'mine' 'is' '1' '/' '16' '"' 'off'
 '.' '.' '.' 'and' 'there' 'is' 'no' 'way' 'to' 'adjust' 'it' '.' 'plus'
 'you' 'have' 'to' '"' 'recalibrate' '"' 'it' 'if' 'it' 'sits' 'for'
 'more' 'then' 'a' 'few' 'days' '-' 'which' 'i' 'actually' 'would' 'not'
 'mind' ',' 'if' 'it' 'calibrated' 'to' 'be' 'accurate' '!' 'so' ','
 'every' 'time' 'i' 'use' 'the' 'digital' 'part' ',' 'i' 'have' 'to' 'add'
 '1' '/' '16' 'to' 'what' 'ever' 'the' 'screen' 'says' '.' '.' '.'
 'considering' 'i' 'bought' 'it' 'for' 'the' '"' 'digital' 'accuracy' '"'
 'that' 'is' 'totally' 'annoying' '.' 'since' 'it' 'is' 'useless' 'as' 'a'
 'digital' 'measure' '-' '$' '35' 'is' 'a' 'ridiculous' 'price' 'for' 'a'
 'standard' 'tape' 'measure' 'that' 'looks' 'cool' '.' 'i' 'will' 'not'
 'be' 'buying' 'another' ',' 'nor' 'will' 'i' 'recommend' 'to' 'anyone'
 '.' 'in' 'fact' ',' 'i' 'will' 'be' 't

In [2]:
import pandas as pd

df = pd.read_parquet(
    "datasets/ner/ner_train_small.parquet"
)

entity_tokens = 0
total_tokens = 0

for labels in df["ner_tags"]:

    labels = labels.tolist()

    total_tokens += len(labels)

    entity_tokens += sum(

        label != "O"

        for label in labels

    )

print("Total tokens :", total_tokens)
print("Entity tokens:", entity_tokens)

print(
    "Entity ratio:",
    entity_tokens / total_tokens
)

Total tokens : 41929
Entity tokens: 421
Entity ratio: 0.010040783228791528


In [4]:
from collections import Counter
import pandas as pd

df = pd.read_parquet("datasets/ner/ner_train_small.parquet")

counter = Counter()

for labels in df["ner_tags"]:
    labels = labels.tolist()
    counter.update(labels)

print(counter)

Counter({'O': 41508, 'B-PRODUCT': 185, 'B-SUBCATEGORY': 124, 'I-PRODUCT': 49, 'B-CATEGORY': 43, 'I-SUBCATEGORY': 11, 'B-DEPARTMENT': 8, 'I-CATEGORY': 1})


In [5]:
from src.analytics.aspect_engine import AspectEngine
from src.analytics.models import Entity

entities = [

    Entity(
        text="battery",
        label="COMPONENT",
        confidence=0.98,
    ),

    Entity(
        text="drill",
        label="PRODUCT",
        confidence=0.99,
    ),

]

aspects = AspectEngine.extract(

    review="The battery is terrible but the drill is excellent.",

    sentiment="negative",

    sentiment_confidence=0.93,

    entities=entities,

)

for aspect in aspects:

    print(aspect)

Aspect(entity='battery', entity_type='COMPONENT', sentiment='negative', confidence=0.93)
Aspect(entity='drill', entity_type='PRODUCT', sentiment='negative', confidence=0.93)
